# knot — 02: ingest

Bind a source to a class. See the write SQL knot emits. Bind row
data to it via the connector. Verify what landed.

In [1]:
import json
import uuid

import psycopg

from knot import Spec, types

In [2]:
# Single class + one source binding. New surface compared to 01:
#   * `spec.add_source("imdb")` — declares an external system.
#   * `imdb.bind(movie)` — declares which class imdb publishes.
#   * `.set_default_weight(0.85)` — weight is the resolver's argmax
#     key. Opaque float; higher wins. 0.85 means "trust imdb more
#     than an unweighted source".
spec = Spec(identifier_slot_name="canonical_id")

movie = spec.add_class("Movie")
movie.slot("title", types.TEXT, required=True)
movie.slot("year", types.INTEGER)

imdb = spec.add_source("imdb")
movie_b = imdb.bind(movie).set_default_weight(0.85)
movie_b

SourceBinding(source=Source(name='imdb', description=None), class_=OntologyClass(name='Movie', kind=<ClassKind.CONCRETE: 'concrete'>, is_a=None, mixins=[], slots=[Slot(name='canonical_id', type=<Primitive.TEXT: 'text'>, identifier=True, required=True, description=None), Slot(name='title', type=<Primitive.TEXT: 'text'>, identifier=False, required=True, description=None), Slot(name='year', type=<Primitive.INTEGER: 'integer'>, identifier=False, required=False, description=None)], description=None), default_weight=0.85, slot_mappings={}, slot_weights={}, description=None)

In [3]:
# Host plumbing + deploy. Schema name is a throwaway per-run id.
pg = psycopg.connect(
    host="localhost", port=5433,
    user="knot", password="knot", dbname="knot",
    autocommit=True,
)
schema = f"knot_play_{uuid.uuid4().hex[:8]}"
pg.execute(spec.init_sql(schema=schema))
schema

'knot_play_e88f79f4'

In [4]:
# A batch of two rows from imdb. Each row carries:
#   * `source_identifier` — imdb's own ID for the movie (their key).
#   * `canonical_id` — knot's cross-source identity. Pre-assigned
#     here (synchronous ER); a separate notebook will cover the
#     async path where canonical_id starts NULL and gets assigned
#     by an ER worker later.
#   * the class slots (`title`, `year`) as native values.
rows = [
    {
        "source_identifier": "tt0110912",
        "canonical_id": "m_pulpfiction",
        "title": "Pulp Fiction",
        "year": 1994,
    },
    {
        "source_identifier": "tt2878306",
        "canonical_id": "m_killbill1",
        "title": "Kill Bill: Vol. 1",
        "year": 2003,
    },
]
rows

[{'source_identifier': 'tt0110912',
  'canonical_id': 'm_pulpfiction',
  'title': 'Pulp Fiction',
  'year': 1994},
 {'source_identifier': 'tt2878306',
  'canonical_id': 'm_killbill1',
  'title': 'Kill Bill: Vol. 1',
  'year': 2003}]

In [5]:
# `binding.write_sql()` returns two SQL templates — both reference a
# single `%(rows)s::jsonb` parameter. knot never touches the rows;
# the host's connector binds them at execute time.
close_out, insert = movie_b.write_sql(schema=schema)
print("--- close_out ---")
print(close_out)
print("\n--- insert ---")
print(insert)

--- close_out ---
UPDATE knot_play_e88f79f4.movie_bindings AS b
SET valid_to = now()
FROM (
    SELECT
        (r->>'canonical_id') AS canonical_id,
        (r->>'source_identifier') AS source_identifier
    FROM jsonb_array_elements(%(rows)s::jsonb) AS r
) AS keys
WHERE b.canonical_id = keys.canonical_id
  AND b.source_name = 'imdb'
  AND b.source_identifier = keys.source_identifier
  AND b.valid_to IS NULL;

--- insert ---
INSERT INTO knot_play_e88f79f4.movie_bindings (source_name, source_identifier, canonical_id, title, year, raw_payload)
SELECT
    'imdb',
    raw.source_identifier,
    raw.canonical_id::text,
    raw.title::text,
    raw.year::integer,
    raw.__raw_payload
FROM (
    SELECT
        r AS __raw_payload,
        (r->>'source_identifier') AS source_identifier,
        (r->>'canonical_id') AS canonical_id,
        (r->>'title') AS title,
        (r->>'year') AS year
    FROM jsonb_array_elements(%(rows)s::jsonb) AS r
) AS raw;


In [6]:
# Run both statements with the rows bound as a single jsonb param.
# For an autocommit connection each cur.execute commits independently;
# wrap in pg.transaction() if you want atomic close_out + insert.
payload = json.dumps(rows)
with pg.cursor() as cur:
    cur.execute(close_out, {"rows": payload})
    cur.execute(insert, {"rows": payload})

In [7]:
# Verify via a knot Query against the resolved view — what the
# user-facing read API sees. Each cur.fetchone()-shape result row
# is the merged (this notebook: single-source = pass-through)
# per-canonical_id state.
q = movie.order_by(movie.col.year).select(
    movie.col.canonical_id, movie.col.title, movie.col.year
)
sql, params = q.sql(schema=schema)
with pg.cursor() as cur:
    cur.execute(sql, params or None)
    cols = [d.name for d in cur.description]
    for row in cur.fetchall():
        print(dict(zip(cols, row)))

{'canonical_id': 'm_pulpfiction', 'title': 'Pulp Fiction', 'year': 1994}
{'canonical_id': 'm_killbill1', 'title': 'Kill Bill: Vol. 1', 'year': 2003}
